In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
url = "https://github.com/dissectingDreams/customer_personality_analysis/blob/main/marketing_campaign.csv"
data = pd.read_csv(url, sep = "\t")

ParserError: Error tokenizing data. C error: Expected 1 fields in line 1294, saw 28001


In [ ]:
data.head()

In [ ]:
data.isnull().sum()                 # количество пропусков в каждой колонке

В колонке доходов видим, что есть 24 пропуска, остальные колонки заполнены полностью

In [ ]:
data.describe()

In [ ]:
sns.boxplot(x=data['Year_Birth'])

Чтобы сделать опеределение возраста в дальнейшем более наглядным и удобным, заменим год рождения на возраст

 По каждому клиенту у нас есть дата регистрации в Dt_Customer, будет полезно использовать эту информацию как точку расчета его возраста

In [ ]:
import datetime

#т.к. дата изначально не в том формате что нужен модулю datetime, переводим её в d-m-Y
data['Dt_Customer'] = pd.to_datetime(data['Dt_Customer'], format='%d-%m-%Y', errors='coerce')

#удаляем строки, где дата не преобразовалась
data = data.dropna(subset=['Dt_Customer'])
data['Age'] = data['Dt_Customer'].dt.year - data['Year_Birth']

#Year_Birth и Dt_Customer больше не понадобятся, для чистоты таблицы их можно удалить
data.drop('Year_Birth', axis=1, inplace=True)
data.drop('Dt_Customer', axis=1, inplace=True)

#Перемещаем Age в удобное место, после Teenhome
age_col = data.pop('Age')
data.insert(6, 'Age', age_col)

Т.к. нам неинтересна информация о том, в какую из маркетинговых кампаний откликнулся клиент, объединим все эти колонки в одну, которая покажет, откликнулся ли клиент вообще

In [ ]:
#список колонок, которые объединим
campaign_cols = ['Response', 'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']

#создаём новую колонку — 1, если хотя бы одна из них равна 1
data['overall_response'] = data[campaign_cols].any(axis=1).astype(int)

#удаляем лишние колонки
data.drop('AcceptedCmp1', axis=1, inplace=True)
data.drop('AcceptedCmp2', axis=1, inplace=True)
data.drop('AcceptedCmp3', axis=1, inplace=True)
data.drop('AcceptedCmp4', axis=1, inplace=True)
data.drop('AcceptedCmp5', axis=1, inplace=True)
data.drop('Response', axis=1, inplace=True)

#ID также ни на что не влияет, от него можно избавиться
data.drop("ID", axis=1, inplace=True)

In [ ]:
#объединю колонки "Kidhome" и "Teenhome" в одну "Childhome" для удобства

data['Childhome'] = data['Teenhome'] + data['Kidhome']
data.drop('Kidhome', axis=1, inplace=True)
data.drop('Teenhome', axis=1, inplace=True)

children_col = data.pop('Childhome')
data.insert(4, 'Childhome', children_col)

In [ ]:
#Z_CostContact и Z_Revenue технические переменные, в нашей работе бесполезны, удаляем
data.drop(['Z_CostContact', 'Z_Revenue'], axis=1, inplace=True)

In [ ]:
data.head()

Далее я занялся поиском и обработкой выбросов, в данном случае наиболее удобным инструментом оказался "ящик с усами"

In [ ]:
# Только числовые колонки
numeric_cols = data.select_dtypes(include=['number']).columns

for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data[col])
    plt.title(f'Распределение: {col}')
    plt.show()

In [ ]:
#самый верхний выброс дохода кажется нереалистичным и его лучше отсеять
data = data[data['Income'] <= 200000]
#возраст регистрации выше 100 также является нереалистичным исключением
data = data[data['Age'] <= 100]

Остальные данные пока что трогать не будем, т.к. они могут быть полезны для сегментации

Далее создадим таблицу сравнения значений для каждой группы потребителей и общего значения в целом. Для числовых значений будем использовать медиану, а для категориальных - моду.

In [ ]:
# Создаём колонки-маркеры
data['wine_buyer'] = data['MntWines'] > 0
data['fruits_buyer'] = data['MntFruits'] > 0
data['meat_buyer'] = data['MntMeatProducts'] > 0
data['fish_buyer'] = data['MntFishProducts'] > 0
data['sweet_buyer'] = data['MntSweetProducts'] > 0
data['gold_buyer'] = data['MntGoldProds'] > 0

numeric_features = ['Income', 'Age', 'MntWines', 'MntFruits', 'MntMeatProducts', 
                     'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']

categorical_features = ['Education', 'Marital_Status', 'Childhome']

#функция для создания таблицы
def summary_by_group(data, group_col, numeric_features, categorical_features):
    result = {}
    
    # Для числовых — медиана
    for col in numeric_features:
        result[col] = data.groupby(group_col)[col].median()
    
    # Для категориальных — мода
    for col in categorical_features:
        result[col] = data.groupby(group_col)[col].agg(lambda x: x.mode()[0] if not x.mode().empty else None)
    
    return pd.DataFrame(result).T

products = {
    'wine_buyer': 'Покупатели вина',
    'fruits_buyer': 'Покупатели фруктов',
    'meat_buyer': 'Покупатели мяса',
    'fish_buyer': 'Покупатели рыбы',
    'sweet_buyer': 'Покупатели сладостей',
    'gold_buyer': 'Покупатели золота'
}

all_results = {}

for col, label in products.items():
    summary = summary_by_group(data, col, numeric_features, categorical_features)
    all_results[label] = summary[True]

#сборка таблицы
all_clients_median = data[numeric_features].median()
all_clients_mode = data[categorical_features].agg(lambda x: x.mode()[0] if not x.mode().empty else None)
all_clients = pd.concat([all_clients_median, all_clients_mode])

final_table = pd.DataFrame(all_results)

#добавляем колонку по всем клиентам и переставляем её в начало
final_table['Все клиенты'] = all_clients
cols = ['Все клиенты'] + [col for col in final_table.columns if col != 'Все клиенты']
final_table = final_table[cols]

final_table

В целом по таблице выше уже можно сделать вывод о целевом портрете потребителя. Для наглядности проведём визуализацию разных метрик

In [ ]:
plot_data = final_table.loc[numeric_features].copy().astype(float)

#Сохраняем базовый столбец
baseline = plot_data['Все клиенты']

# 3. Вычитаем базовый столбец из ВСЕХ (включая его самого)
plot_data_diff = plot_data.subtract(baseline, axis=0)

# 4. Теперь в столбце «Все клиенты» — нули.
#    Заменяем их на исходные значения (медианы)
plot_data_diff['Все клиенты'] = baseline.values

# 5. Строим тепловую карту
plt.figure(figsize=(12, 8))
sns.heatmap(plot_data_diff, 
            annot=True, 
            fmt='.0f', 
            cmap='RdBu_r', 
            center=0,
            vmin=-10,
            vmax=250,
            cbar_kws={'label': 'Отклонение от среднего по всем клиентам'},
            linewidths=0.5,
            linecolor='white')

plt.title('Сравнение групп покупателей\n'
          '(столбец "Все клиенты" — базовые значения, остальные — отклонения)', 
          fontsize=14)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

То что практически в каждой ячейке показатели выше среднего по всем клиентам это нормально, т.к. группы часто пересекаются и происходит подсчет дважды, однако это не меняет картины отклонения в целом

In [ ]:
# Создаём список групп
buyer_cols = ['wine_buyer', 'fruits_buyer', 'meat_buyer', 'fish_buyer', 'sweet_buyer', 'gold_buyer']
labels = ['Вино', 'Фрукты', 'Мясо', 'Рыба', 'Сладости', 'Золото']

plt.figure(figsize=(12, 6))

#Собираем данные в один DataFrame
violin_data = pd.DataFrame()

#ограничим значения дохода, для более красивой визуализации

data = data[data['Income'] <= 110000]

for col, label in zip(buyer_cols, labels):
    temp = data[data[col] == True][['Income']].copy()
    temp['Группа'] = label
    violin_data = pd.concat([violin_data, temp])

#Строим violin plot
sns.violinplot(data=violin_data, 
               x='Группа', 
               y='Income', 
               hue='Группа', 
               inner='quartile', 
               palette='Set2', 
               legend=False)

По пунктирным линиям видно, что у покупателей фруктов распределение немного смещено вверх, однако в целом распределения очень похожи друг на друга, значит доход не является сильным разделителем по группам

In [ ]:
# Создаём сводную таблицу: образование по каждой группе (в процентах)
products = {
    'wine_buyer': 'Вино',
    'fruits_buyer': 'Фрукты',
    'meat_buyer': 'Мясо',
    'fish_buyer': 'Рыба',
    'sweet_buyer': 'Сладости',
    'gold_buyer': 'Золото'
}

edu_summary = {}

for col, label in products.items():
    # Процент Graduation среди покупателей
    edu_summary[label] = data[data[col] == True]['Education'].value_counts(normalize=True) * 100

# Объединяем в один DataFrame
edu_df = pd.DataFrame(edu_summary).fillna(0)

# Визуализация
edu_df.T.plot(kind='bar', figsize=(12, 6), edgecolor='black')
plt.title('Распределение образования по группам покупателей')
plt.ylabel('Процент')
plt.xlabel('Группа')
plt.xticks(rotation=45)
plt.legend(title='Образование')
plt.tight_layout()
plt.show()